## Step 1: Dataset load karna

In [1]:
!pip install -q datasets scikit-learn pandas

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Zainabsa99/mitre_attack", split="train")
print("Dataset loaded successfully!")
print(dataset)

df = dataset.to_pandas()
df

Dataset loaded successfully!
Dataset({
    features: ['ID', 'name', 'description', 'created', 'domain', 'tactics', 'detection', 'text'],
    num_rows: 508
})


,ID,name,description,created,domain,tactics,detection,text
0,T1578.005,Modify Cloud Compute Infrastructure: Modify Cl...,Adversaries may modify settings that directly ...,05 September 2023,enterprise-attack,Defense Evasion,None,
1,T1071.003,Application Layer Protocol: Mail Protocols,Adversaries may communicate using application ...,15 March 2020,enterprise-attack,Command and Control,Analyze network data for uncommon data flows (...,
2,T1055.003,Process Injection: Thread Execution Hijacking,Adversaries may inject malicious code into hij...,14 January 2020,enterprise-attack,"Defense Evasion, Privilege Escalation",Monitoring Windows API calls indicative of the...,
3,T1550,Use Alternate Authentication Material,Adversaries may use alternate authentication m...,30 January 2020,enterprise-attack,"Defense Evasion, Lateral Movement","Configure robust, consistent account activity ...",
4,T1584.003,Compromise Infrastructure: Virtual Private Server,Adversaries may compromise third-party Virtual...,01 October 2020,enterprise-attack,Resource Development,Once adversaries have provisioned software on ...,
...,...,...,...,...,...,...,...,...
503,T1546.002,Event Triggered Execution: Screensaver,Adversaries may establish persistence by execu...,24 January 2020,enterprise-attack,"Persistence, Privilege Escalation",Monitor process execution and command-line par...,
504,T1560.001,Archive Collected Data: Archive via Utility,Adversaries may use utilities to compress and/...,20 February 2020,enterprise-attack,Collection,Common utilities that may be present on the sy...,
505,T1078.001,Valid Accounts: Default Accounts,Adversaries may obtain and abuse credentials o...,13 March 2020,enterprise-attack,"Defense Evasion, Initial Access, Persistence, ...",Monitor whether default accounts have been act...,
506,T1069.003,Permission Groups Discovery: Cloud Groups,Adversaries may attempt to find cloud groups a...,21 February 2020,enterprise-attack,Discovery,System and network discovery techniques normal...,


In [17]:
df.to_csv('mitre_attack.csv', index=False)

## Step 2: Basic preprocessing [Data Cleaning)

In [2]:
#Data Cleaning

df = df.copy()

df["description"] = df["description"].fillna("")
df["detection"] = df["detection"].fillna("")
df["tactics"] = df["tactics"].fillna("")
df["name"] = df["name"].fillna("")
df["ID"] = df["ID"].fillna("")

# Remove completely empty descriptions
df = df[df["description"].str.strip() != ""]

print(df.shape)



(508, 8)


## Step 3: Instruction /Response Format

In [3]:
examples = []

for _, row in df.iterrows():

    technique_id = row["ID"]
    name = row["name"]
    description = row["description"]
    tactics = row["tactics"]
    detection = row["detection"]

    # ------------------------------------------------
    # 1. Technique identification
    # ------------------------------------------------
    examples.append({
        "instruction": f"What is MITRE ATT&CK technique {technique_id}?",
        "response": (
            f"{technique_id} is {name}.\n\n"
            f"{description}"
        )
    })

    # ------------------------------------------------
    # 2. Description
    # ------------------------------------------------
    examples.append({
        "instruction": f"Explain MITRE ATT&CK technique {technique_id}.",
        "response": description
    })

    # ------------------------------------------------
    # 3. Tactic
    # ------------------------------------------------
    examples.append({
        "instruction": f"What tactic or tactics does {technique_id} belong to?",
        "response": (
            f"{technique_id} ({name}) belongs to the following "
            f"MITRE ATT&CK tactic(s): {tactics}."
        )
    })

    # ------------------------------------------------
    # 4. Detection
    # ------------------------------------------------
    if detection.strip():
        examples.append({
            "instruction": (
                f"How can defenders detect MITRE ATT&CK technique "
                f"{technique_id} ({name})?"
            ),
            "response": detection
        })

    # ------------------------------------------------
    # 5. Name -> explanation
    # ------------------------------------------------
    examples.append({
        "instruction": f"What is {name} in MITRE ATT&CK?",
        "response": (
            f"{name} ({technique_id}) is a MITRE ATT&CK technique "
            f"associated with the {tactics} tactic(s).\n\n"
            f"{description}"
        )
    })

    # ------------------------------------------------
    # 6. Full summary
    # ------------------------------------------------
    full_response = (
        f"Technique ID: {technique_id}\n"
        f"Technique: {name}\n"
        f"Tactic(s): {tactics}\n\n"
        f"Description:\n{description}"
    )

    if detection.strip():
        full_response += f"\n\nDetection:\n{detection}"

    examples.append({
        "instruction": f"Give me a complete MITRE ATT&CK summary for {technique_id}.",
        "response": full_response
    })

print("Number of examples:", len(examples))


Number of examples: 3006


## Step 4: Generate HuggingFace Dataset 

In [4]:
from datasets import Dataset

formatted_dataset = Dataset.from_list(examples)

print(formatted_dataset)
print(formatted_dataset[0])


Dataset({
    features: ['instruction', 'response'],
    num_rows: 3006
})
{'instruction': 'What is MITRE ATT&CK technique T1578.005?', 'response': 'T1578.005 is Modify Cloud Compute Infrastructure: Modify Cloud Compute Configurations.\n\nAdversaries may modify settings that directly affect the size, locations, and resources available to cloud compute infrastructure in order to evade defenses. These settings may include service quotas, subscription associations, tenant-wide policies, or other configurations that impact available compute. Such modifications may allow adversaries to abuse the victim’s compute resources to achieve their goals, potentially without affecting the execution of running instances and/or revealing their activities to the victim.\n\nFor example, cloud providers often limit customer usage of compute resources via quotas. Customers may request adjustments to these quotas to support increased computing needs, though these adjustments may require approval from the cl

In [5]:
## Step 5: Train/validation split

split_dataset = formatted_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training:", len(train_dataset))
print("Validation:", len(eval_dataset))


Training: 2555
Validation: 451


## Step 6: Load Base model + tokenizer

In [6]:
from unsloth import FastLanguageModel

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060. Num GPUs = 1. Max memory: 7.614 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


## Step 7: Llama-3 chat template apply karna

In [7]:
def formatting_func(example):

    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }


## Step 8: LoRA adapters lagana

In [8]:
model = FastLanguageModel.get_peft_model(
    model,

    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha=16,
    lora_dropout=0,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=3407,

    use_rslora=False,
)


Unsloth 2026.8.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [9]:
train_dataset = train_dataset.map(
    formatting_func,
    remove_columns=train_dataset.column_names
)

eval_dataset = eval_dataset.map(
    formatting_func,
    remove_columns=eval_dataset.column_names
)
print(train_dataset[0]["text"])


Map:   0%|          | 0/2555 [00:00<?, ? examples/s]

Map:   0%|          | 0/451 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Explain MITRE ATT&CK technique T1560.002.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

An adversary may compress or encrypt data that is collected prior to exfiltration using 3rd party libraries. Many libraries exist that can archive data, including [Python](https://attack.mitre.org/techniques/T1059/006) rarfile (Citation: PyPI RAR), libzip (Citation: libzip), and zlib (Citation: Zlib Github). Most libraries include functionality to encrypt and/or compress data.

Some archival libraries are preinstalled on systems, such as bzip2 on macOS and Linux, and zip on Windows. Note that the libraries are different from the utilities. The libraries can be linked against when compiling, while the utilities require spawning a subshell, or a similar execution mechanism.<|eot_id|>


## Step 9: Trainer setup

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,

    args=TrainingArguments(
        output_dir="./mitre-llama3",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,       # <-- important
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,                           # <-- if your GPU supports FP16
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        seed=3407,
        report_to="none",
        load_best_model_at_end=True,
    ),
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2555 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/451 [00:00<?, ? examples/s]

## Step 11: Start Training

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,555 | Num Epochs = 3 | Total steps = 960
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,1.887596,1.662401
100,1.401341,1.395597
150,1.417983,1.224845
200,1.196105,1.078723
250,1.051813,0.995341
300,1.003360,0.883941
350,0.606415,0.765670
400,0.583351,0.691935
450,0.478212,0.610882
500,0.438410,0.535596


Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-350/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3/checkpoint-450/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3

## Step 11: Inference test — controlled test

In [19]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "What is MITRE ATT&CK technique T1047.001?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=True,
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


user

What is MITRE ATT&CK technique T1047.001?assistant

T1047.001 is Windows Management Instrumentation: Local WMI.

Adversaries may use Windows Management Instrumentation (WMI) to execute malicious commands. WMI is designed for programmers and is the infrastructure for management data and operations on Windows systems.(Citation: WMI 1-3) WMI is an administration feature that provides a uniform environment to access Windows system components.

The WMI service enables both local and remote access, though the latter is facilitated by [Remote Services](https://attack.mitre.org/techniques/T1021) such as [Distributed Component Object Model](https://attack.mitre.org/techniques/T1021/003) and [Windows Remote Management](https://attack.mitre.org/techniques/T1021/006).(Citation: WMI 1-3) Remote WMI over DCOM operates using port 135, whereas WMI over WinRM operates over port 5985 when using HTTP and 5986 for HTTPS.(Citation: WMI 1-3) (Citation: Mandiant WMI)

An adversary can use WMI to intera

## Step 12: Save Model

In [15]:
model.save_pretrained("./mitre-llama3-lora")
tokenizer.save_pretrained("./mitre-llama3-lora")


Unsloth: Restored added_tokens_decoder metadata in ./mitre-llama3-lora/tokenizer_config.json.


('./mitre-llama3-lora/tokenizer_config.json',
 './mitre-llama3-lora/chat_template.jinja',
 './mitre-llama3-lora/tokenizer.json')

In [1]:
!nvidia-smi

Failed to initialize NVML: Driver/library version mismatch
NVML library version: 580.178


In [2]:
python -c "import torch; print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('Device count:', torch.cuda.device_count())"


SyntaxError: invalid syntax (3168645519.py, line 1)

In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.10.0+cu130
PyTorch CUDA: 13.0
CUDA available: False
GPU count: 0


/home/znb/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/znb/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [1]:
import os

# Fragmentation avoid karne ke liye
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from unsloth import FastLanguageModel

max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mitre-llama3-lora",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060. Num GPUs = 1. Max memory: 7.614 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load mitre-llama3-lora as a legacy tokenizer.
Unsloth 2026.8.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded successfully!
GPU: NVIDIA GeForce RTX 4060
Allocated: 5.49 GB


In [4]:
import csv
import json
import torch
from collections import defaultdict, Counter

TIMESERIES_CSV = "mitre_timeseries-11.csv"
OUT_JSON = "mitre_utility_report_lora-3.json"

def load_unique_techniques(path):
    technique_counts = Counter()
    technique_ips = defaultdict(set)
    technique_names = {}
    tactic_of_technique = {}

    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            tid = row.get("technique_id")
            if not tid:
                continue
            technique_counts[tid] += 1
            technique_ips[tid].add(row.get("ip"))
            technique_names[tid] = row.get("technique_name", "")
            tactic_of_technique[tid] = row.get("tactic_id", "")

    return technique_counts, technique_ips, technique_names, tactic_of_technique

def ask_local_model(model, tokenizer, technique_id, max_new_tokens=256, temperature=0.2):
    messages = [
        {"role": "user", "content": f"What is MITRE ATT&CK technique {technique_id}?"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
        
    new_tokens = output_ids[0][inputs.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

def build_utility_report(model, tokenizer, technique_counts, technique_ips, technique_names, tactic_of_technique):
    report = []
    for tid in technique_counts:
        print(f"\nQuerying local LoRA model for {tid} ...")
        try:
            explanation = ask_local_model(model, tokenizer, tid)
            print("Prediction:", explanation)
        except Exception as e:
            print(f"  ERROR querying {tid}: {e}")
            explanation = None

        report.append({
            "technique_id": tid,
            "technique_name": technique_names.get(tid, ""),
            "tactic_id": tactic_of_technique.get(tid, ""),
            "occurrences_this_week": technique_counts[tid],
            "unique_ips_involved": sorted(technique_ips[tid]),
            "model_explanation": explanation,
        })
    return report


def main():
    technique_counts, technique_ips, technique_names, tactic_of_technique = load_unique_techniques(TIMESERIES_CSV)
    print(f"Total unique MITRE techniques: {len(technique_counts)}")
    for tid, cnt in technique_counts.most_common():
        print(f"  {tid} -> {cnt} occurrences, {len(technique_ips[tid])} unique IP(s)")

    report = build_utility_report(model, tokenizer, technique_counts, technique_ips, technique_names, tactic_of_technique)

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print(f"\nTotal {len(report)} unique techniques ka report save ho gaya: {OUT_JSON}")


main()  

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Total unique MITRE techniques: 2
  T1590 -> 114 occurrences, 3 unique IP(s)
  T1046 -> 30 occurrences, 2 unique IP(s)

Querying local LoRA model for T1590 ...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prediction: T1590 is Gather Victim Network Information.

Adversaries may gather information about the victim's networks that can be used during targeting. Information about networks may include a variety of details, including administrative data (ex: IP ranges, domain names, etc.) as well as specifics regarding its topology and operations.

Adversaries may gather this information in various ways, such as direct collection actions via [Active Scanning](https://attack.mitre.org/techniques/T1595) or [Phishing for Information](https://attack.mitre.org/techniques/T1598). Information about networks may also be exposed to adversaries via online or other accessible data sets (ex: [Search Open Technical Databases](https://attack.mitre.org/techniques/T1596)).(Citation: WHOIS)(Citation: DNS Dumpster)(Citation: Circl Passive DNS) Gathering this information may reveal opportunities for other forms of reconnaissance (ex: [Active Scanning](https://attack.mitre.org/techniques/T1595) or [Search Open We

In [7]:
messages = [
    {
        "role": "user",
        "content": "What is MITRE ATT&CK technique T1048?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        temperature=0.2,
        do_sample=True,
    )

input_length = inputs.shape[1]

response = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True
)

print("\nPrediction:")
print(response)


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prediction:
T1048 is Exfiltration Over Alternative Protocol.

Adversaries may steal data by exfiltrating it over a different protocol than that of the existing command and control channel. The data may also be sent to an alternate network location from the main command and control server.  

Alternate protocols include FTP, SMTP, HTTP/S, DNS, SMB, or any other network protocol not being used as the main command and control channel. Adversaries may also opt to encrypt and/or obfuscate these alternate channels. 

[Exfiltration Over Alternative Protocol](https://attack.mitre.org/techniques/T1048) can be done


In [2]:
model.save_pretrained_gguf(
    "/home/znb/Documents/mitre-llama3-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Downloading `unsloth/llama-3-8b-instruct` into the Hugging Face cache so future exports skip the 15.0GB download...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `/home/znb/Documents/mitre-llama3-gguf`: 
Unsloth: Copying 4 files from cache to `/home/znb/Documents/mitre-llama3-gguf`: 


Cache copy failed: Error copying model-00002-of-00004.safetensors from /home/znb/.cache/huggingface/hub/models--unsloth--llama-3-8b-instruct/snapshots/f3710969eb766fb49d4d1ed3aeabcb03390772bd/model-00002-of-00004.safetensors to /home/znb/Documents/mitre-llama3-gguf: Failed to copy /home/znb/.cache/huggingface/hub/models--unsloth--llama-3-8b-instruct/snapshots/f3710969eb766fb49d4d1ed3aeabcb03390772bd/model-00002-of-00004.safetensors to /home/znb/Documents/mitre-llama3-gguf/model-00002-of-00004.safetensors: [Errno 122] Disk quota exceeded: '/home/znb/.cache/huggingface/hub/models--unsloth--llama-3-8b-instruct/snapshots/f3710969eb766fb49d4d1ed3aeabcb03390772bd/model-00002-of-00004.safetensors' -> '/home/znb/Documents/mitre-llama3-gguf/model-00002-of-00004.safetensors'
Failed to copy one or more files from cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. 


Unsloth: Preparing safetensor model files:  50%|█ | 2/4 [00:00<00:00,  2.36it/s]


RuntimeError: Failed to save/merge model: [Errno 122] Disk quota exceeded